# 🎯 Notebook 03 — Classification: Churn Prediction
Models: Random Forest, XGBoost, Logistic Regression

Covers: training, evaluation, ROC curves, confusion matrices, SHAP explainability

In [ ]:
import sys
sys.path.insert(0, '../src')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from data.loader import load_raw_data
from data.preprocessor import prepare_data
from models.classifier import build_models, get_feature_importance
from models.evaluator import (
    plot_roc_curves, plot_confusion_matrix,
    plot_feature_importance, results_summary_table
)

df = load_raw_data()
X_train, X_test, y_train, y_test, preprocessor = prepare_data(df)
print('Data ready.')

## 1. Train All Models

In [ ]:
from models.classifier import train_and_evaluate
results, best_name = train_and_evaluate(X_train, X_test, y_train, y_test)
print(f'\nBest model: {best_name}')

## 2. Performance Summary

In [ ]:
summary = results_summary_table(results)
print(summary.to_string(index=False))

## 3. ROC Curves

In [ ]:
fig = plot_roc_curves(results, y_test)
plt.tight_layout()
plt.show()

## 4. Confusion Matrix (Best Model)

In [ ]:
best_res = results[best_name]
fig = plot_confusion_matrix(y_test, best_res['y_pred'], model_name=best_name)
plt.tight_layout()
plt.show()

## 5. Feature Importance

In [ ]:
fi = get_feature_importance(best_res['model'], list(X_train.columns))
fig = plot_feature_importance(fi, top_n=15)
plt.tight_layout()
plt.show()

print('\nTop 10 features:')
print(fi.head(10).to_string(index=False))

## 6. SHAP Explainability

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(best_res['model'])
    shap_values = explainer.shap_values(X_test)

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test, plot_type='bar',
                      feature_names=list(X_test.columns), show=False)
    plt.title('SHAP Feature Importance', fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Beeswarm / dot plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_test,
                      feature_names=list(X_test.columns), show=False)
    plt.title('SHAP Value Distribution', fontweight='bold')
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f'SHAP unavailable: {e}\nInstall with: pip install shap')

## 7. Threshold Analysis

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score

y_prob = best_res['y_prob']
thresholds = np.arange(0.3, 0.9, 0.05)
rows = []
for t in thresholds:
    y_pred_t = (y_prob >= t).astype(int)
    rows.append({
        'Threshold': f'{t:.2f}',
        'Precision': precision_score(y_test, y_pred_t, zero_division=0),
        'Recall':    recall_score(y_test, y_pred_t),
        'F1':        f1_score(y_test, y_pred_t),
        'Flagged':   y_pred_t.sum(),
    })

thresh_df = pd.DataFrame(rows)
print(thresh_df.to_string(index=False))